In [0]:
""" 
Identify the amount of Trips that were made on each day and its corresponding fare and tip amount that was collected 
Also pull the corresponding AQI value for that day in the city 
"""

df = spark.sql("""
                with AQI_Data as (
                select Date , max(AQI) as MAQI from demo.silver.nyc_aqi
                Group by Date
                )
                select 
                cast(pickup_datetime as date) as pickup_date, 
                Cast(sum(total_amount) as decimal(10,2)) as TotalAmount, 
                Cast(sum(tip_amount) as decimal(10,2)) as TotalTipAmount, 
                count(1) as Trip_Count , 
                max(MAQI) as M_AQI
                from demo.silver.nyc_taxi  A join 
                aqi_data B on cast(A.pickup_datetime as date) = B.Date
                group by cast(pickup_datetime as date)
                order by max(MAQI) desc 
               """)



In [0]:
"""
Create a Temp view on the Dataframe and insert the data into Gold Layer table 
"""
df.createOrReplaceTempView("Temp_Taxi_Trip_AQI")

# If the data already exists, update the data into the table else insert data into the table 
spark.sql("""
          Merge into demo.gold.taxi_trips_aqi A 
          Using Temp_Taxi_Trip_AQI B 
          on A.pickup_date = B.pickup_date  
          when matched then
          Update Set A.total_amount = B.TotalAmount, 
          A.total_tip_amount = B.TotalTipAmount, 
          A.trip_count = B.Trip_Count, 
          A.aqi = B.M_AQI
          when not matched 
          then insert (pickup_date, total_amount, total_tip_amount, trip_count, aqi) 
          values (B.pickup_date, B.TotalAmount, B.TotalTipAmount, B.Trip_Count, B.M_AQI)    
          """)


